In [ ]:
# =============================================================================
# Deep Reinforcement Learning for Structural Damage Detection
# Algorithm: Soft Actor-Critic (SAC)
# =============================================================================
#
# Research Objective
# ------------------
# This implementation trains a Soft Actor-Critic (SAC) agent to estimate
# story-wise structural damage indices from acceleration response signals
# of a multi-story shear building model.
#
# Input (State)
# -------------
# Normalized acceleration time series from each floor sensor.
#
# Action
# ------
# Continuous vector representing estimated damage ratio for each floor.
# Bounds: [0.5 , 1.0]
#
# Reward
# ------
# Negative mean absolute error between predicted damage vector and
# true damage pattern.
#
# Dataset
# -------
# RL_Dataset.zip
#     ├── scenario_x.csv
#     └── labels.json
#
# This notebook is designed as a clean research pipeline suitable for
# academic reproducibility and GitHub publication.
# =============================================================================



# =============================================================================
# 1. Install Required Libraries (Colab)
# =============================================================================
!pip install stable-baselines3[extra] shimmy scikit-learn --quiet



# =============================================================================
# 2. Import Libraries
# =============================================================================
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import gymnasium as gym
from gymnasium import spaces

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from stable_baselines3 import SAC



# =============================================================================
# 3. Upload Dataset
# =============================================================================
from google.colab import files

uploaded = files.upload()

with zipfile.ZipFile("RL_Dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/RL_Dataset")

dataset_dir = "/content/RL_Dataset"

print("Dataset extracted successfully.")



# =============================================================================
# 4. Load Scenario Labels
# =============================================================================
labels_path = os.path.join(dataset_dir, "labels.json")

with open(labels_path, "r") as f:
    labels_data = json.load(f)

print("Number of scenarios:", len(labels_data))

scaler = MinMaxScaler(feature_range=(-1, 1))



# =============================================================================
# 5. Custom Reinforcement Learning Environment
# =============================================================================
class DamageDetectionEnv(gym.Env):

    """
    Custom environment for structural damage detection using
    reinforcement learning.

    Observation:
        Flattened normalized acceleration signals

    Action:
        Continuous vector representing damage level per floor

    Reward:
        Negative MAE between predicted and true damage
    """

    def __init__(self, dataset_dir, labels):

        super().__init__()

        self.dataset_dir = dataset_dir
        self.labels = labels

        self.num_floors = 5
        self.num_samples = 100

        self.observation_space = spaces.Box(
            low=-1,
            high=1,
            shape=(self.num_floors * self.num_samples,),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=0.5,
            high=1.0,
            shape=(self.num_floors,),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):

        idx = np.random.randint(0, len(self.labels))
        label_info = self.labels[idx]

        file_path = os.path.join(
            self.dataset_dir,
            f"{label_info['scenario']}.csv"
        )

        data = pd.read_csv(file_path).values[:self.num_samples, :]

        scaled_data = scaler.fit_transform(data)

        self.current_observation = scaled_data.flatten().astype(np.float32)
        self.current_label = np.array(label_info["damage_pattern"])

        return self.current_observation, {}

    def step(self, action):

        error = np.abs(action - self.current_label)

        reward = -np.mean(error)

        terminated = True
        truncated = False

        info = {
            "true_damage": self.current_label.tolist(),
            "predicted": action.tolist()
        }

        return self.current_observation, reward, terminated, truncated, info



# =============================================================================
# 6. Environment Initialization
# =============================================================================
env = DamageDetectionEnv(dataset_dir, labels_data)



# =============================================================================
# 7. SAC Model Configuration
# =============================================================================
policy_kwargs = dict(
    net_arch=[256, 256],
    activation_fn=torch.nn.ReLU
)

model = SAC(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-5,
    batch_size=64,
    train_freq=1,
    gradient_steps=1,
    learning_starts=1000,
    buffer_size=100000,
    policy_kwargs=policy_kwargs
)



# =============================================================================
# 8. Training Phase
# =============================================================================
model.learn(total_timesteps=100000)

print("Training completed.")



# =============================================================================
# 9. Evaluation Phase
# =============================================================================
results = []

for _ in range(20):

    obs, _ = env.reset()

    action, _ = model.predict(obs, deterministic=True)

    _, _, _, _, info = env.step(action)

    results.append(info)

print("Evaluation completed.")



# =============================================================================
# 10. Error Visualization
# =============================================================================
true_vals = np.array([r["true_damage"] for r in results])
pred_vals = np.array([r["predicted"] for r in results])

errors = np.abs(true_vals - pred_vals)

plt.figure(figsize=(10,5))

plt.boxplot(errors, labels=[f"Floor {i+1}" for i in range(5)])

plt.title("Absolute Prediction Error of SAC Model per Floor")
plt.ylabel("|Predicted − True|")

plt.grid(True)

plt.show()



# =============================================================================
# 11. Quantitative Performance Metrics
# =============================================================================
y_true = true_vals.flatten()
y_pred = pred_vals.flatten()

mae = mean_absolute_error(y_true, y_pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))

accuracy = np.mean(np.abs(y_true - y_pred) < 0.05) * 100

print("SAC Performance Metrics")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"Accuracy (|error| < 0.05): {accuracy:.2f}%")
